In [1]:
# Core
import numpy as np
import pandas as pd

# Visualization 
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
#StratifiedKFold is cross validation
#gridsearchcv is hyperparameter tuning
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB


# Metrics
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score,recall_score,f1_score,precision_score
from sklearn.metrics import confusion_matrix, classification_report


c:\Users\Dilsh\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
#import moels in models folder using joblib
x_train = joblib.load('../Models/x_train.pkl')
y_train = joblib.load('../Models/y_train.pkl')
x_test = joblib.load('../Models/x_test.pkl')
y_test = joblib.load('../Models/y_test.pkl')


In [5]:

models = {
    "Logistic": Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),

    "SVM": Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearSVC(dual=False))
    ]),

    "DecisionTree": Pipeline([
        ('model', DecisionTreeClassifier())
    ]),

    "RandomForest": Pipeline([
        ('model', RandomForestClassifier(random_state=42))
    ]),

    "LightGBM": Pipeline([
        ('model', LGBMClassifier(random_state=42))
    ]),

    "CatBoost": Pipeline([
        ('model', CatBoostClassifier(verbose=0))
    ]),

    "NaiveBayes": Pipeline([
        ('scaler', StandardScaler()),
        ('model', GaussianNB())
    ])
}

In [6]:
params = {
    "Logistic": {
        'model__C': [0.1, 1, 10]# C is the regularization parameter in logistic regression. It controls the strength of regularization. A smaller C value means stronger regularization, while a larger C value means weaker regularization.
    },

    "SVM": {
        'model__C': [0.1, 1, 10]# C is the regularization parameter in SVM. It controls the trade-off between achieving a low error on the training data and minimizing the norm of the weights. A smaller C value creates a wider margin, while a larger C value creates a narrower margin.
    },

    "DecisionTree": {
        'model__max_depth': [3, 5, 10]# max_depth is the maximum depth of the decision tree. It limits how deep the tree can grow. A smaller max_depth can help prevent overfitting, while a larger max_depth can allow the model to capture more complex patterns in the data.
    },

    "RandomForest": {
        'model__n_estimators': [100, 200],
        'model__max_depth': [5, 10]
    },

    "LightGBM": {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.01, 0.1]
    },

    "CatBoost": {
        'model__iterations': [100, 200],
        'model__depth': [4, 6]
    },

    "NaiveBayes": {
        'model__var_smoothing': [1e-9, 1e-8]
    }
}

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#overfiting like kata padan
#underfiting like  can get idea  like

In [8]:

results = []
best_models = {}

for name in models:
    print(f"\n Training {name}...")

    grid = GridSearchCV(
        models[name],
        param_grid=params[name],
        cv=cv,
        scoring='f1',# F1 score is a good metric for imbalanced datasets as it considers both precision and recall.
        n_jobs=-1
    )

    grid.fit(x_train, y_train)

    best_models[name] = grid.best_estimator_

    # Get scores
    if hasattr(grid.best_estimator_, "predict_proba"):
        prob = grid.best_estimator_.predict_proba(x_test)[:, 1]
    else:
        prob = grid.best_estimator_.decision_function(x_test)

    #  Threshold tuning
    best_f1 = 0
    best_thresh = 0.5

    for t in np.arange(0.3, 0.7, 0.05):
        y_pred = (prob > t).astype(int)
        f1 = f1_score(y_test, y_pred)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    results.append({
        "Model": name,
        "Best_F1": best_f1,
        "Threshold": best_thresh,
        "Best_Params": grid.best_params_
    })

    print("Best F1:", best_f1)
    print("Best Threshold:", best_thresh)


 Training Logistic...
Best F1: 0.8711724575598921
Best Threshold: 0.44999999999999996

 Training SVM...
Best F1: 0.8361420843992267
Best Threshold: 0.3

 Training DecisionTree...
Best F1: 0.8700131804515813
Best Threshold: 0.39999999999999997

 Training RandomForest...
Best F1: 0.8724896872454655
Best Threshold: 0.44999999999999996

 Training LightGBM...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 225963, number of negative: 278037
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017655 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 418
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448339 -> initscore=-0.207383
[LightGBM] [Info] Start training from score -0.207383
Be

c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
11 fits failed out of a total of 20.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
11 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit

Best F1: 0.8771754143646409
Best Threshold: 0.39999999999999997

 Training NaiveBayes...
Best F1: 0.8588201428808192
Best Threshold: 0.39999999999999997


In [ ]:
#save all models
for name, model in best_models.items():
    joblib.dump(model, f'../model/{name}_model.pkl')
    

##### GET All Models Using Joblib

In [10]:
#get all models 
import os

folder_path = "../Models/with_out_pca_models"


model_list = []

for file in os.listdir(folder_path):
    if file.endswith(".pkl"):
        model = joblib.load(os.path.join(folder_path, file))
        # check if it is a model
        if hasattr(model, "predict"):
            model_list.append({
                "Model_Name": file.replace(".pkl", ""),
                "Model_Object": model
            })

# check loaded models
for m in model_list:
    print(m["Model_Name"])

CatBoost_model
DecisionTree_model
LightGBM_model
Logistic_model
NaiveBayes_model
RandomForest_model
SVM_model


In [ ]:
# create a DataFrame from the list of models
df_models = pd.DataFrame(model_list)
# show the DataFrame
print(df_models)

           Model_Name                                       Model_Object
0      CatBoost_model  (CatBoostClassifier(depth=4, iterations=200, v...
1  DecisionTree_model             (DecisionTreeClassifier(max_depth=10))
2      LightGBM_model  (LGBMClassifier(n_estimators=200, random_state...
3      Logistic_model  (StandardScaler(), LogisticRegression(C=1, max...
4    NaiveBayes_model                   (StandardScaler(), GaussianNB())
5  RandomForest_model  ((DecisionTreeClassifier(max_depth=10, max_fea...
6           SVM_model   (StandardScaler(), LinearSVC(C=0.1, dual=False))


In [12]:
X_test = joblib.load("../Models/x_test.pkl")
y_test = joblib.load("../Models/y_test.pkl")

print(X_test.shape, y_test.shape)

(125932, 13) (125932,)


In [ ]:
# Evaluate all models on the test set
results = []

for i, row in df_models.iterrows():
    model = row["Model_Object"]
    name = row["Model_Name"]

    y_pred = model.predict(X_test)

    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Recall": recall,
        "Best_F1": f1
    })

In [ ]:
# Create a DataFrame to display results
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Best_F1", ascending=False, ignore_index=True)

print(results_df)

                Model  Accuracy    Recall   Best_F1
0      CatBoost_model  0.889901  0.869183  0.876235
1      LightGBM_model  0.889226  0.869236  0.875578
2  RandomForest_model  0.885764  0.861638  0.871204
3      Logistic_model  0.884088  0.859177  0.869236
4           SVM_model  0.883850  0.857725  0.868810
5  DecisionTree_model  0.882135  0.858008  0.867168
6    NaiveBayes_model  0.872082  0.858043  0.857459


In [ ]:
# For example, add a default threshold 0.5 if you don't have one
results_df['Threshold'] = 0.5

# Create a dict of models for easy access
best_models = {row['Model_Name']: row['Model_Object'] for idx, row in df_models.iterrows()}

# Evaluate each model with threshold
for i, row in results_df.iterrows():
    model_name = row["Model"]
    threshold = row["Threshold"]
    model = best_models[model_name]

    print("\n" + "="*50)
    print(f"🔹 MODEL: {model_name}")
    print("="*50)

    # Get probabilities or decision scores
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_test)[:, 1]
    else:
        prob = model.decision_function(X_test)

    # Apply threshold
    y_pred = (prob > threshold).astype(int)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")

    # Confusion Matrix
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


🔹 MODEL: CatBoost_model
Accuracy : 0.8899
Precision: 0.8834
Recall   : 0.8692
F1 Score : 0.8762

Confusion Matrix:
[[62986  6478]
 [ 7387 49081]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.91      0.90     69464
           1       0.88      0.87      0.88     56468

    accuracy                           0.89    125932
   macro avg       0.89      0.89      0.89    125932
weighted avg       0.89      0.89      0.89    125932


🔹 MODEL: LightGBM_model
Accuracy : 0.8892
Precision: 0.8820
Recall   : 0.8692
F1 Score : 0.8756

Confusion Matrix:
[[62898  6566]
 [ 7384 49084]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.91      0.90     69464
           1       0.88      0.87      0.88     56468

    accuracy                           0.89    125932
   macro avg       0.89      0.89      0.89    125932
weighted avg       0.89      0.89      0.89    125932


🔹 M

In [ ]:
# Identify the best model based on F1 score
best_model_name = results_df.iloc[0]["Model"]
best_model = best_models[best_model_name]

print(" BEST MODEL:", best_model_name)

 BEST MODEL: CatBoost_model


In [ ]:
# Get the best threshold for the best model
best_thresh = results_df.iloc[0]["Threshold"]

if hasattr(best_model, "predict_proba"):
    prob = best_model.predict_proba(X_test)[:, 1]
else:
    prob = best_model.decision_function(X_test)

y_pred = (prob > best_thresh).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Accuracy: 0.8899008988978179
Recall: 0.8691825458666855
Precision: 0.8834032289998021
F1: 0.8762351933016148


#### With PCA 

In [18]:
# lode this model  in to this one
pca = joblib.load("../Models/pca_model.pkl")
X_test= joblib.load("../Models/x_test_pca.pkl")
y_test = joblib.load("../Models/y_test.pkl")  
X_train= joblib.load("../Models/x_train_pca.pkl")
y_train = joblib.load("../Models/y_train.pkl")

In [19]:

models = {
    "Logistic": Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),

    "SVM": Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearSVC(dual=False))
    ]),

    "DecisionTree": Pipeline([
        ('model', DecisionTreeClassifier())
    ]),

    "RandomForest": Pipeline([
        ('model', RandomForestClassifier(random_state=42))
    ]),

    "LightGBM": Pipeline([
        ('model', LGBMClassifier(random_state=42))
    ]),

    "CatBoost": Pipeline([
        ('model', CatBoostClassifier(verbose=0))
    ]),

    "NaiveBayes": Pipeline([
        ('scaler', StandardScaler()),
        ('model', GaussianNB())
    ])
}


In [20]:
params = {
    "Logistic": {
        'model__C': [0.1, 1, 10]
    },

    "SVM": {
        'model__C': [0.1, 1, 10]
    },

    "DecisionTree": {
        'model__max_depth': [3, 5, 10]
    },

    "RandomForest": {
        'model__n_estimators': [100, 200],
        'model__max_depth': [5, 10]
    },

    "LightGBM": {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.01, 0.1]
    },

    "CatBoost": {
        'model__iterations': [100, 200],
        'model__depth': [4, 6]
    },

    "NaiveBayes": {
        'model__var_smoothing': [1e-9, 1e-8]
    }
}


In [21]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [22]:
results = []
best_models = {}

for name in models:
    print(f"\n Training {name}...")

    grid = GridSearchCV(
        models[name],
        param_grid=params[name],
        cv=cv,
        scoring='f1',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_

    # Get scores
    if hasattr(grid.best_estimator_, "predict_proba"):
        prob = grid.best_estimator_.predict_proba(X_test)[:, 1]
    else:
        prob = grid.best_estimator_.decision_function(X_test)

    #  Threshold tuning
    best_f1 = 0
    best_thresh = 0.5

    for t in np.arange(0.3, 0.7, 0.05):
        y_pred = (prob > t).astype(int)
        f1 = f1_score(y_test, y_pred)

        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t

    results.append({
        "Model": name,
        "Best_F1": best_f1,
        "Threshold": best_thresh,
        "Best_Params": grid.best_params_
    })

    print("Best F1:", best_f1)
    print("Best Threshold:", best_thresh)



 Training Logistic...
Best F1: 0.8365247065717585
Best Threshold: 0.39999999999999997

 Training SVM...
Best F1: 0.7876891183447287
Best Threshold: 0.3

 Training DecisionTree...
Best F1: 0.8392499700603925
Best Threshold: 0.39999999999999997

 Training RandomForest...
Best F1: 0.8460685052370428
Best Threshold: 0.44999999999999996

 Training LightGBM...
[LightGBM] [Info] Number of positive: 225873, number of negative: 277855
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007150 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 503728, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.448403 -> initscore=-0.207127
[LightGBM] [Info] Start training from score -0.207127


c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Best F1: 0.8591552920788257
Best Threshold: 0.39999999999999997

 Training CatBoost...


c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
9 fits failed out of a total of 20.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
8 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\Dilsh\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    self._final_estimator.fit(X

Best F1: 0.8628416257883672
Best Threshold: 0.44999999999999996

 Training NaiveBayes...
Best F1: 0.8208401390459438
Best Threshold: 0.35


In [23]:
# Define the folder where you want to save models
save_folder = "../Models/with_pca_models"

# Create folder if it doesn't exist
os.makedirs(save_folder, exist_ok=True)
for name, model in best_models.items():
    file_path = os.path.join(save_folder, f"{name}_pca_model.pkl")
    joblib.dump(model, file_path)
    print(f" Saved {name} at {file_path}")

 Saved Logistic at ../Models/with_pca_models\Logistic_pca_model.pkl
 Saved SVM at ../Models/with_pca_models\SVM_pca_model.pkl
 Saved DecisionTree at ../Models/with_pca_models\DecisionTree_pca_model.pkl
 Saved RandomForest at ../Models/with_pca_models\RandomForest_pca_model.pkl
 Saved LightGBM at ../Models/with_pca_models\LightGBM_pca_model.pkl
 Saved CatBoost at ../Models/with_pca_models\CatBoost_pca_model.pkl
 Saved NaiveBayes at ../Models/with_pca_models\NaiveBayes_pca_model.pkl
